# IRRM-CODEC: run and analyze

This notebook shows how to train from two inputs: an AIRR table and a parquet file with TCRemP embeddings.

## 1. Define inputs

The AIRR file must contain `clone_id`, `junction_aa`, `v_call`, `j_call`, `locus`.
The embeddings parquet must contain `clone_id` plus either `tcremp_emb` or numeric embedding columns.

In [1]:
chain='trg'

In [2]:
from pathlib import Path

root = Path.cwd().resolve().parent if Path.cwd().name == 'notebooks' else Path.cwd().resolve()
airr_path = f'/projects/immunestatus/vdjrearm/airr_format/{chain}_background_100k.tsv' #root / 'data' / 'sample_airr.tsv'
embeddings_path = f'/projects/immunestatus/vdjrearm/tcremp/{chain}_background_100k_tcremp.parquet'
output_dir = root / 'artifacts' / f'forward_demo_{chain}'

airr_path, embeddings_path, output_dir

('/projects/immunestatus/vdjrearm/airr_format/trg_background_100k.tsv',
 '/projects/immunestatus/vdjrearm/tcremp/trg_background_100k_tcremp.parquet',
 PosixPath('/home/evlasova/irrm-codec/artifacts/forward_demo_trg'))

In [3]:
import sys
sys.path.insert(0, str(root))

## 2. Preview the two files

In [4]:
import pandas as pd

airr_df = pd.read_csv(airr_path, sep='\t')
emb_df = pd.read_parquet(embeddings_path)

display(airr_df.head())
display(emb_df.head())

,junction_aa,v_call,j_call,locus
0,CATWARSDWIKTF,TRGV2,TRGJP2,gamma
1,CATWVHYKKLF,TRGV3,TRGJ2,gamma
2,CATWDLLAGHYKKLF,TRGV5,TRGJ1,gamma
3,CATWDEEKLF,TRGV2,TRGJ2,gamma
4,CATWFYLYKKLF,TRGV3,TRGJ1,gamma


/home/evlasova/.conda/envs/irrm-codec/lib/python3.11/site-packages/pandas/io/formats/format.py:1466: RuntimeWarning: overflow encountered in cast
  has_large_values = (abs_vals > 1e6).any()


,0_g_v,0_g_j,0_g_cdr3,1_g_v,1_g_j,1_g_cdr3,2_g_v,2_g_j,2_g_cdr3,3_g_v,...,2996_g_cdr3,2997_g_v,2997_g_j,2997_g_cdr3,2998_g_v,2998_g_j,2998_g_cdr3,2999_g_v,2999_g_j,2999_g_cdr3
0,324.0,75.0,690.0,0.0,238.0,710.0,0.0,0.0,370.0,91.0,...,560.0,899.0,238.0,910.0,0.0,259.0,1120.0,277.0,0.0,430.0
1,110.0,252.0,1050.0,320.0,21.0,490.0,320.0,259.0,930.0,315.0,...,1120.0,891.0,21.0,490.0,320.0,0.0,820.0,349.0,259.0,1010.0
2,0.0,247.0,990.0,324.0,0.0,1030.0,324.0,238.0,1030.0,351.0,...,1040.0,881.0,0.0,610.0,324.0,21.0,620.0,337.0,238.0,850.0
3,324.0,252.0,870.0,0.0,21.0,450.0,0.0,259.0,790.0,91.0,...,1000.0,899.0,21.0,790.0,0.0,0.0,920.0,277.0,259.0,810.0
4,110.0,247.0,1140.0,320.0,0.0,560.0,320.0,238.0,1040.0,315.0,...,1150.0,891.0,0.0,760.0,320.0,21.0,890.0,349.0,238.0,1080.0


## 3. Launch training

In [5]:
import subprocess

cmd = [
    sys.executable, "-m", "irrm_codec.train_forward",
    "--airr-path", str(airr_path),
    "--embeddings-path", str(embeddings_path),
    "--output-dir", str(output_dir),
    "--locus", "gamma",
    "--epochs", "40",
    "--lr", "3e-4",
    "--weight-decay", "1e-3",
    "--dropout", "0.2",
    "--encoder-type", "plain_conv",
    "--hidden-dim", "192",
    "--mlp-dim", "512",
    "--mlp-hidden-dim", "1024",
    "--dilations", "1,2,4,8",
    "--early-stopping-patience", "5",
    "--scheduler", "plateau",
    "--scheduler-patience", "2",
    "--scheduler-factor", "0.5",
]


subprocess.run(cmd, cwd=root, check=True)
cmd

2026-04-04 03:14:27,391 | INFO | starting forward training
2026-04-04 03:14:27,393 | INFO | output_dir=/home/evlasova/irrm-codec/artifacts/forward_demo_trg
2026-04-04 03:14:27,393 | INFO | device=cpu seed=42
2026-04-04 03:14:27,393 | INFO | hyperparameters batch_size=256 epochs=40 lr=0.000300 weight_decay=0.001000 max_len=30 encoder_type=plain_conv hidden_dim=192 mlp_dim=512 mlp_hidden_dim=1024 dropout=0.200 dilations=1,2,4,8 num_workers=0 log_interval=10 early_stopping_patience=5 scheduler=plateau
Traceback (most recent call last):
  File "<frozen runpy>", line 198, in _run_module_as_main
  File "<frozen runpy>", line 88, in _run_code
  File "/home/evlasova/irrm-codec/irrm_codec/train_forward.py", line 404, in <module>
    main()
  File "/home/evlasova/irrm-codec/irrm_codec/train_forward.py", line 178, in main
    df, emb, merge_stats = load_airr_with_embeddings(
                           ^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/home/evlasova/irrm-codec/irrm_codec/dataio.py", line 132, in

CalledProcessError: Command '['/home/evlasova/.conda/envs/irrm-codec/bin/python', '-m', 'irrm_codec.train_forward', '--airr-path', '/projects/immunestatus/vdjrearm/airr_format/trg_background_100k.tsv', '--embeddings-path', '/projects/immunestatus/vdjrearm/tcremp/trg_background_100k_tcremp.parquet', '--output-dir', '/home/evlasova/irrm-codec/artifacts/forward_demo_trg', '--locus', 'gamma', '--epochs', '40', '--lr', '3e-4', '--weight-decay', '1e-3', '--dropout', '0.2', '--encoder-type', 'plain_conv', '--hidden-dim', '192', '--mlp-dim', '512', '--mlp-hidden-dim', '1024', '--dilations', '1,2,4,8', '--early-stopping-patience', '5', '--scheduler', 'plateau', '--scheduler-patience', '2', '--scheduler-factor', '0.5']' returned non-zero exit status 1.

## 4. Inspect saved metrics and merge stats

In [ ]:
import json
import pandas as pd

history = json.loads((output_dir / 'history.json').read_text())
test_metrics = json.loads((output_dir / 'test_metrics.json').read_text())
data_stats = json.loads((output_dir / 'data_stats.json').read_text())
history_df = pd.json_normalize(history)

display(history_df)
display(test_metrics)
display(data_stats)

In [ ]:
history_df.plot(x='epoch', y=['train.loss', 'val.loss'], figsize=(8, 4), grid=True, title='Loss by epoch')

## 5. Load a checkpoint

In [ ]:
import torch
from irrm_codec.forward_model import ForwardModel

checkpoint = torch.load(output_dir / "best.pt", map_location="cpu")

extra = checkpoint["extra"]
model = ForwardModel(
    output_dim=extra["embedding_dim"],
    max_len=extra.get("max_len", 40),
    hidden_dim=extra.get("hidden_dim", 192),
    mlp_dim=extra.get("mlp_dim", 512),
    mlp_hidden_dim=extra.get("mlp_hidden_dim", 1024),
    dropout=extra.get("dropout", 0.2),
    dilations=tuple(int(x) for x in str(extra.get("dilations", "1,2,4,8")).split(",")),
    encoder_type=extra.get("encoder_type", "residual"),
)

model.load_state_dict(checkpoint["model_state"])
model.eval()

checkpoint["metrics"]


In [ ]:
model.predict('CASSLAPGATNEKLFF')